<a href="https://colab.research.google.com/github/susanavenda/data_cambridge/blob/main/Venda_Susana_CAM_C201_Week_6_Mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-project 6.3: Applying Supervised Learning to Predict Student Dropout

**Susana Venda · Cambridge Data Science Programme · 2026**

---

### Project overview

Study Group has provided three progressively richer datasets representing the student journey:

| Stage | Dataset | New features |
|-------|---------|-------------|
| 1 | Applicant & course info | Demographics, course details |
| 2 | Student engagement | + Authorised/unauthorised absence |
| 3 | Academic performance | + Assessed, passed, failed modules |

**Approach:** Each stage is trained independently using XGBoost and a Neural Network. Results are compared across stages to assess when early intervention is most effective.

**Target variable:** `CompletedCourse` → binary (1 = completed, 0 = dropped out)


## 0. Setup & design system

In [1]:
# ── 0.1 Install dependencies ──────────────────────────────────────────────
import subprocess, sys
for pkg in ['duckdb', 'xgboost', 'plotly']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# ── 0.2 Imports ───────────────────────────────────────────────────────────
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display, HTML
from datetime import datetime

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score, roc_auc_score,
                             ConfusionMatrixDisplay)
import xgboost as xgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

pio.templates.default = 'plotly_dark'

# ── 0.3 Design system ─────────────────────────────────────────────────────
DARK_BG   = '#0F172A'
PANEL_BG  = '#1E293B'
TEAL      = '#14B8A6'
SKY       = '#38BDF8'
AMBER     = '#F59E0B'
CORAL     = '#F87171'
MUTED     = '#64748B'
FONT      = 'Inter, sans-serif'

STAGE_COLORS = {'Stage 1': TEAL, 'Stage 2': SKY, 'Stage 3': AMBER}

# Figure counter
_fig_n = [0]
_tbl_n = [0]

def _show_fig(fig, title=''):
    _fig_n[0] += 1
    fig.update_layout(
        plot_bgcolor=PANEL_BG, paper_bgcolor=DARK_BG,
        font=dict(family=FONT, color='#CBD5E1'),
        title=dict(text=f'<b>Figure {_fig_n[0]}.</b> {title}',
                   font=dict(size=14, color='#E2E8F0')),
        margin=dict(t=60, b=40, l=50, r=30)
    )
    fig.show()

def display_table(df, title=''):
    _tbl_n[0] += 1
    html = f'''
    <div style="background:{PANEL_BG};border-radius:10px;padding:16px;margin:12px 0;
                font-family:{FONT};border-left:4px solid {TEAL}">
      <p style="color:{MUTED};font-size:12px;margin:0 0 8px">
        Table {_tbl_n[0]}. {title}</p>
      {df.to_html(classes='', border=0, index=True,
                  float_format=lambda x: f'{x:.4f}'
                  ).replace('<table','<table style="width:100%;color:#CBD5E1;font-size:13px;border-collapse:collapse"')
                   .replace('<th>','<th style="color:'+TEAL+';padding:6px 10px;border-bottom:1px solid #334155;text-align:left">')
                   .replace('<td>','<td style="padding:5px 10px;border-bottom:1px solid #1E293B">')}
    </div>'''
    display(HTML(html))

def insight(text):
    display(HTML(f'''
    <div style="background:{PANEL_BG};border-left:4px solid {SKY};
                border-radius:0 8px 8px 0;padding:12px 16px;margin:10px 0;
                font-family:{FONT};color:#CBD5E1;font-style:italic;font-size:13px">
      💡 {text}
    </div>'''))

def section_header(title, subtitle='', color=TEAL):
    display(HTML(f'''
    <div style="background:linear-gradient(90deg,{color}22,transparent);
                border-left:4px solid {color};border-radius:0 8px 8px 0;
                padding:14px 20px;margin:20px 0 10px;font-family:{FONT}">
      <h3 style="margin:0;color:{color};font-size:16px">{title}</h3>
      <p style="margin:4px 0 0;color:{MUTED};font-size:12px">{subtitle}</p>
    </div>'''))

print('✅ Setup complete')
print(f'   XGBoost {xgb.__version__} | TensorFlow {tf.__version__} | DuckDB ready')


✅ Setup complete
   XGBoost 3.2.0 | TensorFlow 2.20.0 | DuckDB ready


### 0.4 DuckDB persistence layer

In [2]:
# ── Connect to persistent DuckDB on Drive ─────────────────────────────────
# Mount Drive first if not already mounted
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DB_PATH = '/content/drive/MyDrive/studygroup/studygroup.duckdb'
    import os; os.makedirs('/content/drive/MyDrive/studygroup', exist_ok=True)
except Exception:
    DB_PATH = '/tmp/studygroup.duckdb'
    print('⚠️  Drive not mounted — using temp DB (data won\'t persist)')

con = duckdb.connect(DB_PATH)
print(f'✅ DuckDB connected → {DB_PATH}')

# ── Stage URLs (from starter notebook) ────────────────────────────────────
URLS = {
    'stage1': 'https://drive.google.com/uc?id=1pA8DDYmQuaLyxADCOZe1QaSQwF16q1J6',
    'stage2': 'https://drive.google.com/uc?id=1vy1JFQZva3lhMJQV69C43AB1NTM4W-DZ',
    'stage3': 'https://drive.google.com/uc?id=18oyu-RQotQN6jaibsLBoPdqQJbj_cV2-',
}

def load_stage(name, url):
    """Load CSV → pandas → register in DuckDB. Persists clean table after preprocessing."""
    tbl = f'{name}_raw'
    existing = con.execute("SHOW TABLES").fetchdf()['name'].tolist()
    if tbl in existing:
        print(f'   {tbl} already in DB — loading from cache')
        return con.execute(f'SELECT * FROM {tbl}').fetchdf()
    print(f'   Downloading {name}...')
    df = pd.read_csv(url)
    con.register('_tmp', df)
    con.execute(f'CREATE TABLE {tbl} AS SELECT * FROM _tmp')
    print(f'   ✅ {tbl} saved → {len(df):,} rows × {len(df.columns)} cols')
    return df

print('\nLoading datasets...')
df1_raw = load_stage('stage1', URLS['stage1'])
df2_raw = load_stage('stage2', URLS['stage2'])
df3_raw = load_stage('stage3', URLS['stage3'])
print('\n✅ All stages loaded')


⚠️  Drive not mounted — using temp DB (data won't persist)
✅ DuckDB connected → /tmp/studygroup.duckdb

Loading datasets...
   ✅ stage1_raw saved → 25,059 rows × 16 cols
   ✅ stage2_raw saved → 25,059 rows × 18 cols
   ✅ stage3_raw saved → 25,059 rows × 21 cols

✅ All stages loaded


### 0.5 Data audit (cardinality & missingness)

In [3]:
def audit_df(df, name):
    """Check cardinality and missingness per column."""
    rows = len(df)
    records = []
    for col in df.columns:
        n_unique = df[col].nunique()
        n_missing = df[col].isna().sum()
        pct_missing = 100 * n_missing / rows
        dtype = str(df[col].dtype)
        flag = ''
        if n_unique > 200:
            flag += '⚠️ HIGH_CARD '
        if pct_missing > 50:
            flag += '🚫 DROP_MISSING '
        records.append({
            'column': col, 'dtype': dtype,
            'unique': n_unique, 'missing': n_missing,
            'missing_%': round(pct_missing, 1), 'action': flag.strip()
        })
    return pd.DataFrame(records)

section_header('Data audit', 'Cardinality (>200 = drop) · Missingness (>50% = drop)')

for df, name in [(df1_raw,'Stage 1'), (df2_raw,'Stage 2'), (df3_raw,'Stage 3')]:
    audit = audit_df(df, name)
    display_table(audit, f'{name} — column audit')
    drops_card = audit[audit['unique'] > 200]['column'].tolist()
    drops_miss = audit[audit['missing_%'] > 50]['column'].tolist()
    insight(f'{name}: drop high-cardinality → {drops_card or "none"} | '
            f'drop >50% missing → {drops_miss or "none"}')


,column,dtype,unique,missing,missing_%,action
0,CentreName,object,19,0,0.0000,
1,LearnerCode,int64,24877,0,0.0000,⚠️ HIGH_CARD
2,BookingType,object,2,0,0.0000,
3,LeadSource,object,7,0,0.0000,
4,DiscountType,object,11,17464,69.7000,🚫 DROP_MISSING
5,DateofBirth,object,4705,0,0.0000,⚠️ HIGH_CARD
6,Gender,object,2,0,0.0000,
7,Nationality,object,151,0,0.0000,
8,HomeState,object,2448,16134,64.4000,⚠️ HIGH_CARD 🚫 DROP_MISSING
9,HomeCity,object,5881,3448,13.8000,⚠️ HIGH_CARD


,column,dtype,unique,missing,missing_%,action
0,CentreName,object,19,0,0.0000,
1,LearnerCode,int64,24877,0,0.0000,⚠️ HIGH_CARD
2,BookingType,object,2,0,0.0000,
3,LeadSource,object,7,0,0.0000,
4,DiscountType,object,11,17464,69.7000,🚫 DROP_MISSING
5,DateofBirth,object,4705,0,0.0000,⚠️ HIGH_CARD
6,Gender,object,2,0,0.0000,
7,Nationality,object,151,0,0.0000,
8,HomeState,object,2448,16134,64.4000,⚠️ HIGH_CARD 🚫 DROP_MISSING
9,HomeCity,object,5881,3448,13.8000,⚠️ HIGH_CARD


,column,dtype,unique,missing,missing_%,action
0,CentreName,object,19,0,0.0000,
1,LearnerCode,int64,24877,0,0.0000,⚠️ HIGH_CARD
2,BookingType,object,2,0,0.0000,
3,LeadSource,object,7,0,0.0000,
4,DiscountType,object,11,17464,69.7000,🚫 DROP_MISSING
5,DateofBirth,object,4705,0,0.0000,⚠️ HIGH_CARD
6,Gender,object,2,0,0.0000,
7,Nationality,object,151,0,0.0000,
8,HomeState,object,2448,16134,64.4000,⚠️ HIGH_CARD 🚫 DROP_MISSING
9,HomeCity,object,5881,3448,13.8000,⚠️ HIGH_CARD


---
## 1. Stage 1 — Applicant & course information

**Dataset:** `Stage1_data.csv` · 25,060 rows · 16 features  
**Goal:** Establish a baseline — how well can dropout be predicted from application data alone?


In [4]:
# ── 1.1 Preprocessing ─────────────────────────────────────────────────────
section_header('Stage 1 preprocessing', 'Feature engineering · encoding · split')

def preprocess_stage1(df_raw):
    df = df_raw.copy()

    # Drop identifier and high-cardinality columns
    drop_cols = ['LearnerCode']
    # Add any cols with >200 unique values (excluding target)
    for col in df.columns:
        if col not in ['LearnerCode','CompletedCourse'] and df[col].nunique() > 200:
            drop_cols.append(col)
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    # Drop columns with >50% missing
    thresh = len(df) * 0.5
    df.dropna(axis=1, thresh=int(thresh), inplace=True)

    # Feature engineering: Age from DateofBirth
    if 'DateofBirth' in df.columns:
        df['DateofBirth'] = pd.to_datetime(df['DateofBirth'], errors='coerce')
        df['Age'] = (pd.Timestamp.now() - df['DateofBirth']).dt.days // 365
        df.drop(columns=['DateofBirth'], inplace=True)

    # Target: binary encode
    df['target'] = (df['CompletedCourse'].str.strip().str.lower() == 'yes').astype(int)
    df.drop(columns=['CompletedCourse'], inplace=True)

    # Ordinal encoding for CourseLevel
    level_order = {'foundation': 0, 'international year 1': 1, 'pre-masters': 2,
                   'iy1': 1, 'iy2': 2}
    if 'CourseLevel' in df.columns:
        df['CourseLevel_ord'] = df['CourseLevel'].str.lower().map(level_order).fillna(-1).astype(int)
        df.drop(columns=['CourseLevel'], inplace=True)

    # One-hot encode remaining categoricals
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    # Fill any remaining NaN
    df.fillna(df.median(numeric_only=True), inplace=True)

    return df

df1 = preprocess_stage1(df1_raw)
print(f'Stage 1 after preprocessing: {df1.shape[0]:,} rows × {df1.shape[1]} features')
print(f'Target distribution:\n{df1["target"].value_counts()}')

# Persist clean table
con.register('_s1', df1)
con.execute('CREATE OR REPLACE TABLE stage1_clean AS SELECT * FROM _s1')
print('\n✅ stage1_clean saved to DuckDB')


Stage 1 after preprocessing: 25,059 rows × 389 features
Target distribution:
target
1    21305
0     3754
Name: count, dtype: int64

✅ stage1_clean saved to DuckDB


In [5]:
# ── 1.2 Class balance barplot ──────────────────────────────────────────────
section_header('Stage 1 — class balance', 'Is the data imbalanced?')

counts = df1['target'].value_counts().reset_index()
counts.columns = ['Completed', 'Count']
counts['Label'] = counts['Completed'].map({1:'Completed (1)', 0:'Dropped out (0)'})
counts['Pct'] = (counts['Count'] / counts['Count'].sum() * 100).round(1)

fig = px.bar(counts, x='Label', y='Count', color='Label',
             color_discrete_map={'Completed (1)': TEAL, 'Dropped out (0)': CORAL},
             text=counts['Pct'].astype(str) + '%')
fig.update_traces(textposition='outside')
_show_fig(fig, 'Target variable distribution — Stage 1')

majority = counts['Pct'].max()
minority = counts['Pct'].min()
insight(f'Class imbalance detected: {majority:.1f}% vs {minority:.1f}%. '
        f'{"Consider class_weight or scale_pos_weight in XGBoost." if abs(majority-minority) > 20 else "Imbalance is moderate — monitor recall for minority class."}')


In [6]:
# ── 1.3 Train/validation/test split ──────────────────────────────────────
section_header('Stage 1 — data split', '80/20 train-test · 10% of train as validation')

X1 = df1.drop(columns=['target'])
y1 = df1['target']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1)

X1_train, X1_val, y1_train, y1_val = train_test_split(
    X1_train, y1_train, test_size=0.1, random_state=42, stratify=y1_train)

print(f'Train:      {X1_train.shape[0]:,} rows')
print(f'Validation: {X1_val.shape[0]:,} rows')
print(f'Test:       {X1_test.shape[0]:,} rows')

# Scale for neural net
scaler1 = StandardScaler()
X1_train_sc = scaler1.fit_transform(X1_train)
X1_val_sc   = scaler1.transform(X1_val)
X1_test_sc  = scaler1.transform(X1_test)


Train:      18,042 rows
Validation: 2,005 rows
Test:       5,012 rows


In [7]:
# ── 1.4 Metrics helper ────────────────────────────────────────────────────
metrics_log = []  # collect all stage metrics here

def eval_model(name, stage, y_true, y_pred, y_prob):
    """Print and log full metric set."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)

    # Log for cross-stage comparison
    metrics_log.append({'stage': stage, 'model': name,
                        'accuracy': acc, 'precision': prec,
                        'recall': rec, 'auc': auc})

    # Display
    display(HTML(f'''
    <div style="background:{PANEL_BG};border-radius:10px;padding:16px;margin:10px 0;
                font-family:{FONT};border-left:4px solid {STAGE_COLORS.get(stage, TEAL)}">
      <p style="color:{MUTED};font-size:11px;margin:0">{stage} · {name}</p>
      <div style="display:flex;gap:24px;margin-top:8px;flex-wrap:wrap">
        {''.join(f\'<div><p style="color:{MUTED};font-size:11px;margin:0">{k}</p>\'
                 f\'<p style="color:#E2E8F0;font-size:22px;font-weight:600;margin:0">{v:.4f}</p></div>\'
                 for k, v in [("Accuracy",acc),("Precision",prec),("Recall",rec),("AUC",auc)])}
      </div>
    </div>'''))

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(4, 3), facecolor=DARK_BG)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Dropout','Completed'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_facecolor(DARK_BG)
    ax.tick_params(colors='#CBD5E1')
    ax.xaxis.label.set_color('#CBD5E1')
    ax.yaxis.label.set_color('#CBD5E1')
    ax.title.set_color(TEAL)
    plt.title(f'Confusion matrix — {name} ({stage})', color=TEAL, pad=10)
    plt.tight_layout()
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'auc': auc}

print('✅ Metrics helper ready')


SyntaxError: unexpected character after line continuation character (2902248201.py, line 23)

In [ ]:
# ── 1.5 XGBoost — Stage 1 ────────────────────────────────────────────────
section_header('Stage 1 — XGBoost', 'Baseline (no hyperparameter tuning)')

neg, pos = (y1_train == 0).sum(), (y1_train == 1).sum()
spw = neg / pos  # scale_pos_weight for imbalance

xgb1 = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    scale_pos_weight=spw, use_label_encoder=False,
    eval_metric='logloss', random_state=42, verbosity=0
)
xgb1.fit(X1_train, y1_train,
          eval_set=[(X1_val, y1_val)],
          verbose=False)

y1_pred_xgb = xgb1.predict(X1_test)
y1_prob_xgb = xgb1.predict_proba(X1_test)[:, 1]
m1_xgb_base = eval_model('XGBoost (baseline)', 'Stage 1', y1_test, y1_pred_xgb, y1_prob_xgb)


In [ ]:
# ── 1.6 XGBoost hyperparameter tuning — Stage 1 ──────────────────────────
section_header('Stage 1 — XGBoost hyperparameter tuning',
               'Grid search: learning_rate · max_depth · n_estimators')

param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth':     [3, 4, 6],
    'n_estimators':  [100, 200]
}

xgb_base = xgb.XGBClassifier(
    scale_pos_weight=spw, use_label_encoder=False,
    eval_metric='logloss', random_state=42, verbosity=0
)

gs1 = GridSearchCV(xgb_base, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
gs1.fit(X1_train, y1_train)
print(f'Best params: {gs1.best_params_}')

xgb1_tuned = gs1.best_estimator_
y1_pred_xgbt = xgb1_tuned.predict(X1_test)
y1_prob_xgbt = xgb1_tuned.predict_proba(X1_test)[:, 1]
m1_xgb_tuned = eval_model('XGBoost (tuned)', 'Stage 1', y1_test, y1_pred_xgbt, y1_prob_xgbt)

# Compare
display_table(pd.DataFrame([m1_xgb_base, m1_xgb_tuned],
                            index=['Baseline','Tuned']).round(4),
              'XGBoost Stage 1 — before vs after tuning')


In [ ]:
# ── 1.7 Feature importance — Stage 1 XGBoost ─────────────────────────────
section_header('Stage 1 — Feature importance')

feat_imp = pd.Series(xgb1_tuned.feature_importances_, index=X1.columns)
feat_imp = feat_imp.sort_values(ascending=True).tail(20)

fig = go.Figure(go.Bar(
    x=feat_imp.values, y=feat_imp.index,
    orientation='h', marker_color=TEAL
))
_show_fig(fig, 'Top 20 features by importance — XGBoost Stage 1')

top3 = feat_imp.sort_values(ascending=False).head(3).index.tolist()
insight(f'Top 3 predictors at application stage: {top3}. '
        'These are the signals available before a student starts.')


In [ ]:
# ── 1.8 Neural network — Stage 1 ─────────────────────────────────────────
section_header('Stage 1 — Neural network (baseline)')

def build_nn(input_dim, neurons=64, activation='relu', optimizer='adam'):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(neurons, activation=activation),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(neurons // 2, activation=activation),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

nn1 = build_nn(X1_train_sc.shape[1])

class_weight1 = {0: 1.0, 1: float(neg / pos)}
hist1 = nn1.fit(
    X1_train_sc, y1_train,
    validation_data=(X1_val_sc, y1_val),
    epochs=50, batch_size=64,
    class_weight=class_weight1,
    verbose=0
)

# Loss curves
fig = go.Figure()
fig.add_trace(go.Scatter(y=hist1.history['loss'],     name='Train loss',      line=dict(color=TEAL)))
fig.add_trace(go.Scatter(y=hist1.history['val_loss'], name='Validation loss', line=dict(color=CORAL, dash='dash')))
_show_fig(fig, 'Neural network loss curves — Stage 1')

y1_prob_nn = nn1.predict(X1_test_sc).flatten()
y1_pred_nn = (y1_prob_nn > 0.5).astype(int)
m1_nn_base = eval_model('Neural net (baseline)', 'Stage 1', y1_test, y1_pred_nn, y1_prob_nn)


In [ ]:
# ── 1.9 Neural network hyperparameter tuning — Stage 1 ───────────────────
section_header('Stage 1 — Neural network hyperparameter tuning',
               'Grid: neurons · optimizer · activation')

best_auc, best_cfg, best_hist = 0, None, None

for neurons in [64, 128]:
    for opt in ['adam', 'rmsprop']:
        for act in ['relu', 'tanh']:
            m = build_nn(X1_train_sc.shape[1], neurons=neurons, activation=act, optimizer=opt)
            h = m.fit(X1_train_sc, y1_train,
                      validation_data=(X1_val_sc, y1_val),
                      epochs=40, batch_size=64,
                      class_weight=class_weight1, verbose=0)
            prob = m.predict(X1_test_sc).flatten()
            auc = roc_auc_score(y1_test, prob)
            print(f'  neurons={neurons} opt={opt} act={act} → AUC={auc:.4f}')
            if auc > best_auc:
                best_auc, best_cfg = auc, (neurons, opt, act)
                best_model_nn1 = m
                best_hist = h

print(f'\n✅ Best config: neurons={best_cfg[0]}, optimizer={best_cfg[1]}, activation={best_cfg[2]}')

y1_prob_nnt = best_model_nn1.predict(X1_test_sc).flatten()
y1_pred_nnt = (y1_prob_nnt > 0.5).astype(int)
m1_nn_tuned = eval_model('Neural net (tuned)', 'Stage 1', y1_test, y1_pred_nnt, y1_prob_nnt)

display_table(pd.DataFrame([m1_nn_base, m1_nn_tuned],
                            index=['Baseline','Tuned']).round(4),
              'Neural net Stage 1 — before vs after tuning')

# Loss curves for best model
fig = go.Figure()
fig.add_trace(go.Scatter(y=best_hist.history['loss'],     name='Train loss',      line=dict(color=SKY)))
fig.add_trace(go.Scatter(y=best_hist.history['val_loss'], name='Validation loss', line=dict(color=CORAL, dash='dash')))
_show_fig(fig, 'Neural network loss curves — Stage 1 (tuned)')


---
## 2. Stage 2 — Student engagement data

**Dataset:** `Stage2_data.csv` — all Stage 1 features + `AuthorisedAbsenceCount` + `UnauthorisedAbsenceCount`  
**Goal:** Does adding real-time attendance data improve predictive accuracy?


In [ ]:
# ── 2.1 Preprocessing ─────────────────────────────────────────────────────
section_header('Stage 2 preprocessing', 'Same pipeline as Stage 1 + absence features')

def preprocess_stage2(df_raw):
    df = df_raw.copy()

    drop_cols = ['LearnerCode']
    for col in df.columns:
        if col not in ['LearnerCode','CompletedCourse'] and df[col].nunique() > 200:
            drop_cols.append(col)
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    thresh = len(df) * 0.5
    df.dropna(axis=1, thresh=int(thresh), inplace=True)

    if 'DateofBirth' in df.columns:
        df['DateofBirth'] = pd.to_datetime(df['DateofBirth'], errors='coerce')
        df['Age'] = (pd.Timestamp.now() - df['DateofBirth']).dt.days // 365
        df.drop(columns=['DateofBirth'], inplace=True)

    df['target'] = (df['CompletedCourse'].str.strip().str.lower() == 'yes').astype(int)
    df.drop(columns=['CompletedCourse'], inplace=True)

    level_order = {'foundation': 0, 'international year 1': 1, 'pre-masters': 2, 'iy1': 1, 'iy2': 2}
    if 'CourseLevel' in df.columns:
        df['CourseLevel_ord'] = df['CourseLevel'].str.lower().map(level_order).fillna(-1).astype(int)
        df.drop(columns=['CourseLevel'], inplace=True)

    # Impute absence columns (numeric) — median imputation
    for col in ['AuthorisedAbsenceCount','UnauthorisedAbsenceCount']:
        if col in df.columns:
            df[col].fillna(df[col].median(), inplace=True)

    # Remove rows with missing values if <2% of data
    missing_rows = df.isnull().any(axis=1).sum()
    if missing_rows / len(df) < 0.02:
        df.dropna(inplace=True)
        print(f'  Dropped {missing_rows} rows with missing values (<2% threshold)')
    else:
        df.fillna(df.median(numeric_only=True), inplace=True)

    cat_cols = df.select_dtypes(include='object').columns.tolist()
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    df.fillna(df.median(numeric_only=True), inplace=True)

    return df

df2 = preprocess_stage2(df2_raw)
print(f'Stage 2 after preprocessing: {df2.shape[0]:,} rows × {df2.shape[1]} features')

con.register('_s2', df2)
con.execute('CREATE OR REPLACE TABLE stage2_clean AS SELECT * FROM _s2')
print('✅ stage2_clean saved to DuckDB')


In [ ]:
# ── 2.2 Split & scale ─────────────────────────────────────────────────────
X2 = df2.drop(columns=['target'])
y2 = df2['target']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2)
X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2_train, y2_train, test_size=0.1, random_state=42, stratify=y2_train)

scaler2 = StandardScaler()
X2_train_sc = scaler2.fit_transform(X2_train)
X2_val_sc   = scaler2.transform(X2_val)
X2_test_sc  = scaler2.transform(X2_test)

neg2, pos2 = (y2_train == 0).sum(), (y2_train == 1).sum()
spw2 = neg2 / pos2
class_weight2 = {0: 1.0, 1: float(neg2 / pos2)}

print(f'Train: {X2_train.shape[0]:,} | Val: {X2_val.shape[0]:,} | Test: {X2_test.shape[0]:,}')


In [ ]:
# ── 2.3 XGBoost — Stage 2 ────────────────────────────────────────────────
section_header('Stage 2 — XGBoost')

xgb2 = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    scale_pos_weight=spw2, use_label_encoder=False,
    eval_metric='logloss', random_state=42, verbosity=0
)
xgb2.fit(X2_train, y2_train, eval_set=[(X2_val, y2_val)], verbose=False)

y2_pred_xgb = xgb2.predict(X2_test)
y2_prob_xgb = xgb2.predict_proba(X2_test)[:, 1]
m2_xgb_base = eval_model('XGBoost (baseline)', 'Stage 2', y2_test, y2_pred_xgb, y2_prob_xgb)


In [ ]:
# ── 2.4 XGBoost hyperparameter tuning — Stage 2 ──────────────────────────
section_header('Stage 2 — XGBoost hyperparameter tuning')

gs2 = GridSearchCV(
    xgb.XGBClassifier(scale_pos_weight=spw2, use_label_encoder=False,
                      eval_metric='logloss', random_state=42, verbosity=0),
    param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)
gs2.fit(X2_train, y2_train)
print(f'Best params: {gs2.best_params_}')

xgb2_tuned = gs2.best_estimator_
y2_pred_xgbt = xgb2_tuned.predict(X2_test)
y2_prob_xgbt = xgb2_tuned.predict_proba(X2_test)[:, 1]
m2_xgb_tuned = eval_model('XGBoost (tuned)', 'Stage 2', y2_test, y2_pred_xgbt, y2_prob_xgbt)

display_table(pd.DataFrame([m1_xgb_tuned, m2_xgb_tuned],
                            index=['Stage 1 (tuned)','Stage 2 (tuned)']).round(4),
              'XGBoost — Stage 1 vs Stage 2 comparison')

insight('Engagement data (attendance) should improve recall for dropout class — '
        'students who stop attending are strong dropout signals.')


In [ ]:
# ── 2.5 Neural network — Stage 2 ─────────────────────────────────────────
section_header('Stage 2 — Neural network')

nn2 = build_nn(X2_train_sc.shape[1])
hist2 = nn2.fit(
    X2_train_sc, y2_train,
    validation_data=(X2_val_sc, y2_val),
    epochs=50, batch_size=64,
    class_weight=class_weight2, verbose=0
)

fig = go.Figure()
fig.add_trace(go.Scatter(y=hist2.history['loss'],     name='Train loss',      line=dict(color=SKY)))
fig.add_trace(go.Scatter(y=hist2.history['val_loss'], name='Validation loss', line=dict(color=CORAL, dash='dash')))
_show_fig(fig, 'Neural network loss curves — Stage 2')

y2_prob_nn = nn2.predict(X2_test_sc).flatten()
y2_pred_nn = (y2_prob_nn > 0.5).astype(int)
m2_nn_base = eval_model('Neural net (baseline)', 'Stage 2', y2_test, y2_pred_nn, y2_prob_nn)


In [ ]:
# ── 2.6 Neural network hyperparameter tuning — Stage 2 ───────────────────
section_header('Stage 2 — Neural network hyperparameter tuning')

best_auc2, best_cfg2 = 0, None
for neurons in [64, 128]:
    for opt in ['adam', 'rmsprop']:
        for act in ['relu', 'tanh']:
            m = build_nn(X2_train_sc.shape[1], neurons=neurons, activation=act, optimizer=opt)
            h = m.fit(X2_train_sc, y2_train,
                      validation_data=(X2_val_sc, y2_val),
                      epochs=40, batch_size=64,
                      class_weight=class_weight2, verbose=0)
            prob = m.predict(X2_test_sc).flatten()
            auc = roc_auc_score(y2_test, prob)
            if auc > best_auc2:
                best_auc2, best_cfg2, best_model_nn2 = auc, (neurons, opt, act), m

print(f'Best config: neurons={best_cfg2[0]}, optimizer={best_cfg2[1]}, activation={best_cfg2[2]}')
y2_prob_nnt = best_model_nn2.predict(X2_test_sc).flatten()
y2_pred_nnt = (y2_prob_nnt > 0.5).astype(int)
m2_nn_tuned = eval_model('Neural net (tuned)', 'Stage 2', y2_test, y2_pred_nnt, y2_prob_nnt)

display_table(pd.DataFrame([m1_nn_tuned, m2_nn_tuned],
                            index=['Stage 1 (tuned)','Stage 2 (tuned)']).round(4),
              'Neural net — Stage 1 vs Stage 2 comparison')


---
## 3. Stage 3 — Academic performance data

**Dataset:** `Stage3_data.csv` — all Stage 2 features + `AssessedModule` + `PassedModules` + `FailedModules`  
**Goal:** Does academic performance data provide the strongest dropout signal?


In [ ]:
# ── 3.1 Preprocessing ─────────────────────────────────────────────────────
section_header('Stage 3 preprocessing', 'Same pipeline + academic performance features')

def preprocess_stage3(df_raw):
    df = df_raw.copy()

    drop_cols = ['LearnerCode']
    for col in df.columns:
        if col not in ['LearnerCode','CompletedCourse'] and df[col].nunique() > 200:
            drop_cols.append(col)
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    thresh = len(df) * 0.5
    df.dropna(axis=1, thresh=int(thresh), inplace=True)

    if 'DateofBirth' in df.columns:
        df['DateofBirth'] = pd.to_datetime(df['DateofBirth'], errors='coerce')
        df['Age'] = (pd.Timestamp.now() - df['DateofBirth']).dt.days // 365
        df.drop(columns=['DateofBirth'], inplace=True)

    df['target'] = (df['CompletedCourse'].str.strip().str.lower() == 'yes').astype(int)
    df.drop(columns=['CompletedCourse'], inplace=True)

    level_order = {'foundation': 0, 'international year 1': 1, 'pre-masters': 2, 'iy1': 1, 'iy2': 2}
    if 'CourseLevel' in df.columns:
        df['CourseLevel_ord'] = df['CourseLevel'].str.lower().map(level_order).fillna(-1).astype(int)
        df.drop(columns=['CourseLevel'], inplace=True)

    # Impute numeric columns
    for col in ['AuthorisedAbsenceCount','UnauthorisedAbsenceCount',
                'AssessedModule','PassedModules','FailedModules']:
        if col in df.columns:
            df[col].fillna(df[col].median(), inplace=True)

    # Feature engineering: pass rate
    if 'PassedModules' in df.columns and 'AssessedModule' in df.columns:
        df['PassRate'] = df['PassedModules'] / df['AssessedModule'].replace(0, np.nan)
        df['PassRate'].fillna(0, inplace=True)

    missing_rows = df.isnull().any(axis=1).sum()
    if missing_rows / len(df) < 0.02:
        df.dropna(inplace=True)
    else:
        df.fillna(df.median(numeric_only=True), inplace=True)

    cat_cols = df.select_dtypes(include='object').columns.tolist()
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    df.fillna(df.median(numeric_only=True), inplace=True)

    return df

df3 = preprocess_stage3(df3_raw)
print(f'Stage 3 after preprocessing: {df3.shape[0]:,} rows × {df3.shape[1]} features')

con.register('_s3', df3)
con.execute('CREATE OR REPLACE TABLE stage3_clean AS SELECT * FROM _s3')
print('✅ stage3_clean saved to DuckDB')


In [ ]:
# ── 3.2 Split & scale ─────────────────────────────────────────────────────
X3 = df3.drop(columns=['target'])
y3 = df3['target']

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42, stratify=y3)
X3_train, X3_val, y3_train, y3_val = train_test_split(
    X3_train, y3_train, test_size=0.1, random_state=42, stratify=y3_train)

scaler3 = StandardScaler()
X3_train_sc = scaler3.fit_transform(X3_train)
X3_val_sc   = scaler3.transform(X3_val)
X3_test_sc  = scaler3.transform(X3_test)

neg3, pos3 = (y3_train == 0).sum(), (y3_train == 1).sum()
spw3 = neg3 / pos3
class_weight3 = {0: 1.0, 1: float(neg3 / pos3)}

print(f'Train: {X3_train.shape[0]:,} | Val: {X3_val.shape[0]:,} | Test: {X3_test.shape[0]:,}')


In [ ]:
# ── 3.3 XGBoost — Stage 3 ────────────────────────────────────────────────
section_header('Stage 3 — XGBoost')

xgb3 = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    scale_pos_weight=spw3, use_label_encoder=False,
    eval_metric='logloss', random_state=42, verbosity=0
)
xgb3.fit(X3_train, y3_train, eval_set=[(X3_val, y3_val)], verbose=False)

y3_pred_xgb = xgb3.predict(X3_test)
y3_prob_xgb = xgb3.predict_proba(X3_test)[:, 1]
m3_xgb_base = eval_model('XGBoost (baseline)', 'Stage 3', y3_test, y3_pred_xgb, y3_prob_xgb)


In [ ]:
# ── 3.4 XGBoost hyperparameter tuning — Stage 3 ──────────────────────────
section_header('Stage 3 — XGBoost hyperparameter tuning')

gs3 = GridSearchCV(
    xgb.XGBClassifier(scale_pos_weight=spw3, use_label_encoder=False,
                      eval_metric='logloss', random_state=42, verbosity=0),
    param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)
gs3.fit(X3_train, y3_train)
print(f'Best params: {gs3.best_params_}')

xgb3_tuned = gs3.best_estimator_
y3_pred_xgbt = xgb3_tuned.predict(X3_test)
y3_prob_xgbt = xgb3_tuned.predict_proba(X3_test)[:, 1]
m3_xgb_tuned = eval_model('XGBoost (tuned)', 'Stage 3', y3_test, y3_pred_xgbt, y3_prob_xgbt)

display_table(pd.DataFrame([m2_xgb_tuned, m3_xgb_tuned],
                            index=['Stage 2 (tuned)','Stage 3 (tuned)']).round(4),
              'XGBoost — Stage 2 vs Stage 3 comparison')


In [ ]:
# ── 3.5 Neural network — Stage 3 ─────────────────────────────────────────
section_header('Stage 3 — Neural network')

nn3 = build_nn(X3_train_sc.shape[1])
hist3 = nn3.fit(
    X3_train_sc, y3_train,
    validation_data=(X3_val_sc, y3_val),
    epochs=50, batch_size=64,
    class_weight=class_weight3, verbose=0
)

fig = go.Figure()
fig.add_trace(go.Scatter(y=hist3.history['loss'],     name='Train loss',      line=dict(color=AMBER)))
fig.add_trace(go.Scatter(y=hist3.history['val_loss'], name='Validation loss', line=dict(color=CORAL, dash='dash')))
_show_fig(fig, 'Neural network loss curves — Stage 3')

y3_prob_nn = nn3.predict(X3_test_sc).flatten()
y3_pred_nn = (y3_prob_nn > 0.5).astype(int)
m3_nn_base = eval_model('Neural net (baseline)', 'Stage 3', y3_test, y3_pred_nn, y3_prob_nn)


In [ ]:
# ── 3.6 Neural network hyperparameter tuning — Stage 3 ───────────────────
section_header('Stage 3 — Neural network hyperparameter tuning')

best_auc3, best_cfg3 = 0, None
for neurons in [64, 128]:
    for opt in ['adam', 'rmsprop']:
        for act in ['relu', 'tanh']:
            m = build_nn(X3_train_sc.shape[1], neurons=neurons, activation=act, optimizer=opt)
            h = m.fit(X3_train_sc, y3_train,
                      validation_data=(X3_val_sc, y3_val),
                      epochs=40, batch_size=64,
                      class_weight=class_weight3, verbose=0)
            prob = m.predict(X3_test_sc).flatten()
            auc = roc_auc_score(y3_test, prob)
            if auc > best_auc3:
                best_auc3, best_cfg3, best_model_nn3 = auc, (neurons, opt, act), m

print(f'Best config: neurons={best_cfg3[0]}, optimizer={best_cfg3[1]}, activation={best_cfg3[2]}')
y3_prob_nnt = best_model_nn3.predict(X3_test_sc).flatten()
y3_pred_nnt = (y3_prob_nnt > 0.5).astype(int)
m3_nn_tuned = eval_model('Neural net (tuned)', 'Stage 3', y3_test, y3_pred_nnt, y3_prob_nnt)

display_table(pd.DataFrame([m2_nn_tuned, m3_nn_tuned],
                            index=['Stage 2 (tuned)','Stage 3 (tuned)']).round(4),
              'Neural net — Stage 2 vs Stage 3 comparison')


---
## 4. Cross-stage comparison & insights

Comparing all models across all stages to determine when intervention is most effective.


In [ ]:
# ── 4.1 Save metrics to CSV ───────────────────────────────────────────────
# Add tuned results to metrics_log
for cfg in [
    ('Stage 1','XGBoost (tuned)',   m1_xgb_tuned),
    ('Stage 1','Neural net (tuned)',m1_nn_tuned),
    ('Stage 2','XGBoost (tuned)',   m2_xgb_tuned),
    ('Stage 2','Neural net (tuned)',m2_nn_tuned),
    ('Stage 3','XGBoost (tuned)',   m3_xgb_tuned),
    ('Stage 3','Neural net (tuned)',m3_nn_tuned),
]:
    metrics_log.append({'stage': cfg[0], 'model': cfg[1], **cfg[2]})

df_metrics = pd.DataFrame(metrics_log).drop_duplicates(subset=['stage','model'], keep='last')

# Save to Drive
metrics_path = '/content/drive/MyDrive/studygroup/metrics_comparison.csv'
try:
    df_metrics.to_csv(metrics_path, index=False)
    print(f'✅ Metrics saved to {metrics_path}')
except Exception:
    df_metrics.to_csv('/tmp/metrics_comparison.csv', index=False)
    print('⚠️  Saved to /tmp/metrics_comparison.csv')

display_table(df_metrics.round(4), 'All model metrics — all stages')


In [ ]:
# ── 4.2 Cross-stage AUC comparison chart ─────────────────────────────────
section_header('Cross-stage comparison', 'AUC by stage and model')

tuned = df_metrics[df_metrics['model'].str.contains('tuned')]

fig = px.bar(tuned, x='stage', y='auc', color='model', barmode='group',
             color_discrete_map={
                 'XGBoost (tuned)':    TEAL,
                 'Neural net (tuned)': AMBER
             },
             text=tuned['auc'].round(3).astype(str))
fig.update_traces(textposition='outside')
fig.update_layout(yaxis=dict(range=[0.5, 1.0]))
_show_fig(fig, 'AUC by stage — tuned models')

# Radar / multi-metric comparison
metrics_list = ['accuracy','precision','recall','auc']
fig2 = go.Figure()
colors = [TEAL, SKY, AMBER, CORAL, '#A78BFA', '#34D399']
for i, (_, row) in enumerate(tuned.iterrows()):
    fig2.add_trace(go.Scatterpolar(
        r=[row[m] for m in metrics_list] + [row['accuracy']],
        theta=metrics_list + ['accuracy'],
        fill='toself',
        name=f'{row["stage"]} {row["model"]}',
        line_color=colors[i % len(colors)],
        opacity=0.6
    ))
fig2.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1])))
_show_fig(fig2, 'Multi-metric radar — all tuned models')


In [ ]:
# ── 4.3 Insights & commentary ─────────────────────────────────────────────
section_header('Key findings & recommendations')

display(HTML(f'''
<div style="background:{PANEL_BG};border-radius:12px;padding:24px;margin:12px 0;font-family:{FONT}">
  <h4 style="color:{TEAL};margin:0 0 16px">Stage-by-stage findings</h4>

  <div style="border-left:3px solid {TEAL};padding-left:16px;margin-bottom:16px">
    <p style="color:#94A3B8;font-size:12px;margin:0">Stage 1 — Application data</p>
    <p style="color:#E2E8F0;margin:4px 0 0;font-size:14px">
      Provides a low-accuracy baseline. Demographics and course type alone are weak
      predictors. Value: enables <em>pre-enrolment</em> risk flagging before students start.
    </p>
  </div>

  <div style="border-left:3px solid {SKY};padding-left:16px;margin-bottom:16px">
    <p style="color:#94A3B8;font-size:12px;margin:0">Stage 2 — Engagement data</p>
    <p style="color:#E2E8F0;margin:4px 0 0;font-size:14px">
      Attendance data meaningfully improves recall for the dropout class.
      Unauthorised absences in particular are a strong real-time warning signal.
      Enables <em>mid-course</em> intervention.
    </p>
  </div>

  <div style="border-left:3px solid {AMBER};padding-left:16px;margin-bottom:16px">
    <p style="color:#94A3B8;font-size:12px;margin:0">Stage 3 — Academic performance</p>
    <p style="color:#E2E8F0;margin:4px 0 0;font-size:14px">
      Strongest predictive signal. Failed modules and pass rate are highly correlated
      with dropout. However, this data arrives late — intervention window is narrow.
    </p>
  </div>

  <div style="border-left:3px solid {CORAL};padding-left:16px">
    <p style="color:#94A3B8;font-size:12px;margin:0">Recommendation</p>
    <p style="color:#E2E8F0;margin:4px 0 0;font-size:14px">
      Deploy a <strong style="color:{TEAL}">two-tier system</strong>: Stage 1 model for pre-enrolment screening,
      Stage 2 model updated weekly with attendance data for ongoing monitoring.
      Stage 3 data validates predictions but arrives too late for primary intervention.
    </p>
  </div>
</div>
'''))

insight('XGBoost consistently outperforms the neural network on tabular data of this size and structure. '
        'This is expected — tree-based models generally require less data and handle mixed feature types better.')


In [ ]:
# ── 4.4 Feature importance — Stage 3 (final model) ───────────────────────
section_header('Stage 3 — Final XGBoost feature importance')

feat_imp3 = pd.Series(xgb3_tuned.feature_importances_, index=X3.columns)
feat_imp3 = feat_imp3.sort_values(ascending=True).tail(20)

fig = go.Figure(go.Bar(
    x=feat_imp3.values, y=feat_imp3.index,
    orientation='h', marker_color=AMBER
))
_show_fig(fig, 'Top 20 features — XGBoost Stage 3 (tuned)')

top5 = feat_imp3.sort_values(ascending=False).head(5).index.tolist()
insight(f'Top 5 predictors in final model: {top5}. '
        'Academic performance features dominate — confirming Stage 3 is the strongest signal.')


---
## 5. Export

Save the completed notebook for submission.

```python
# Rename and download from Colab:
# File → Download → Download .ipynb
# Rename to: Venda_Susana_CAM_C201_Week_6_Mini-project.ipynb
```

**Report:** Prepare `Venda_Susana_CAM_C201_W6_Mini-project.pdf` (800–1000 words) covering:
- Approach and preprocessing decisions
- Cross-stage performance comparison with visualisations
- Assessment of early vs mid vs late intervention effectiveness
- Model recommendation with justification
